# 06 First-published against final monthly sitrep
The footnote implies the report tables absorb revisions while CQI does not. The direct test is to compare each month's first-published sitrep with its final revised version, and then ask which of the two CQI agrees with.

I can only test months whose original file I could retrieve from the Internet Archive: April, May, June, October, November and December 2025. July to September and March 2026 originals were not captured, the January 2026 download failed, and February was never revised. I say so rather than fill the gaps.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
final, _ = c.read_msitae_monthly('msitae_monthly')
prov, prov_tot = c.read_msitae_monthly('msitae_provisional')
cqi = c.read_cqi()
src = pd.read_csv(c.BRONZE / 'msitae_provisional' / 'archive_sources.tsv', sep='\t', header=None, names=['file', 'archive_timestamp', 'url'])
for d in (final, prov):
    d['att'] = d[c.ATT_COLS + c.BOOKED_COLS].sum(axis=1)
    d['over4'] = d[c.OVER4_COLS + c.OVER4_BOOKED_COLS].sum(axis=1)
# The provisional December file writes its total row as 'TOTAL ' with a trailing space; read_msitae_monthly strips it, so I check.
print('provisional provider rows sum to their own TOTAL row:', int((prov.groupby('month')[c.ATT_COLS + c.BOOKED_COLS].sum() - prov_tot.set_index('month')[c.ATT_COLS + c.BOOKED_COLS]).abs().max().max()) == 0)
src[['file', 'archive_timestamp']]

provisional provider rows sum to their own TOTAL row: True


,file,archive_timestamp
0,Monthly-AE-April-2025.csv,20250725212946
1,Monthly-AE-May-2025.csv,20250725204415
2,Monthly-AE-June-2025-1-1.csv,20250725205130
3,October-2025-CSV-hg6dl.csv,20260315020951
4,December-2025-CSV-K7F4Sp.csv,20260315021023
5,November-2025-CSV-G9pr3.csv,20260315021010


In [3]:
cq = cqi[(cqi.measure_id == 'AEQI012') & (cqi.org_code != 'ENG')].set_index(['month', 'org_code']).measure_value
rows, changed = [], []
for mth in sorted(prov.month.unique()):
    p = prov[prov.month == mth].set_index('org_code'); f = final[final.month == mth].set_index('org_code')
    j = p[['org_name', 'att', 'over4']].join(f[['att', 'over4']], lsuffix='_prov', rsuffix='_final', how='outer').fillna({'att_prov': 0, 'att_final': 0, 'over4_prov': 0, 'over4_final': 0})
    j['cqi'] = [cq.get((mth, o)) for o in j.index]
    j['revision'] = j.att_final - j.att_prov
    sh = j[j.cqi.notna()]
    rows.append({'month': mth, 'provisional_total': j.att_prov.sum(), 'final_total': j.att_final.sum(), 'revision': j.revision.sum(),
                 'revision_pct': 100 * j.revision.sum() / j.att_prov.sum(), 'providers_revised': int((j.revision != 0).sum()),
                 'four_hour_prov': 100 * (1 - j.over4_prov.sum() / j.att_prov.sum()), 'four_hour_final': 100 * (1 - j.over4_final.sum() / j.att_final.sum()),
                 'cqi_providers': len(sh), 'cqi_equals_final': int((sh.cqi == sh.att_final).sum()), 'cqi_equals_prov': int((sh.cqi == sh.att_prov).sum()),
                 'revised_providers_where_cqi_equals_final': int(((sh.revision != 0) & (sh.cqi == sh.att_final)).sum()),
                 'revised_providers_in_cqi': int((sh.revision != 0).sum())})
    changed.append(j[j.revision != 0].assign(month=mth).reset_index())
res = pd.DataFrame(rows); res['four_hour_change_pp'] = res.four_hour_final - res.four_hour_prov
res.round(3).T

,0,1,2,3,4,5
month,2025-04,2025-05,2025-06,2025-10,2025-11,2025-12
provisional_total,"2,295,820.0000","2,395,886.0000","2,350,478.0000","2,397,701.0000","2,345,860.0000","2,327,015.0000"
final_total,2297421,2397529,2351581,2401266,2347046,2326139
revision,"1,601.0000","1,643.0000","1,103.0000","3,565.0000","1,186.0000",-876.0000
revision_pct,0.0700,0.0690,0.0470,0.1490,0.0510,-0.0380
providers_revised,2,2,1,1,1,3
four_hour_prov,74.7690,75.3940,75.5450,74.1470,74.2030,73.8490
four_hour_final,74.7840,75.4020,75.5560,74.1860,74.2170,73.8420
cqi_providers,153,153,153,153,153,153
cqi_equals_final,153,153,153,153,153,153


In [4]:
revised = pd.concat(changed, ignore_index=True)[['month', 'org_code', 'org_name', 'att_prov', 'att_final', 'revision', 'cqi']]
print('revised provider-months:', len(revised), '| total revision across the six months:', int(revised.revision.sum()))
revised

revised provider-months: 10 | total revision across the six months: 8222


,month,org_code,org_name,att_prov,att_final,revision,cqi
0,2025-04,RBT,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,"9,404.0000",9954,550.0000,"9,954.0000"
1,2025-04,RTD,THE NEWCASTLE UPON TYNE HOSPITALS NHS FOUNDATI...,"19,923.0000",20974,"1,051.0000","20,974.0000"
2,2025-05,RLT,GEORGE ELIOT HOSPITAL NHS TRUST,"8,193.0000",8752,559.0000,"8,752.0000"
3,2025-05,RTD,THE NEWCASTLE UPON TYNE HOSPITALS NHS FOUNDATI...,"20,704.0000",21788,"1,084.0000","21,788.0000"
4,2025-06,RTD,THE NEWCASTLE UPON TYNE HOSPITALS NHS FOUNDATI...,"20,325.0000",21428,"1,103.0000","21,428.0000"
5,2025-10,NL7,NaN,0.0000,3565,"3,565.0000",NaN
6,2025-11,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS F...,"30,799.0000",31985,"1,186.0000","31,985.0000"
7,2025-12,RMP,TAMESIDE AND GLOSSOP INTEGRATED CARE NHS FOUND...,"12,146.0000",11983,-163.0000,"11,983.0000"
8,2025-12,RTG,UNIVERSITY HOSPITALS OF DERBY AND BURTON NHS F...,"30,341.0000",31513,"1,172.0000","31,513.0000"
9,2025-12,RTP,SURREY AND SUSSEX HEALTHCARE NHS TRUST,"12,321.0000",10436,"-1,885.0000","10,436.0000"


In [5]:
# The sharp test: for a provider whose attendances were revised AND that CQI includes, does CQI carry the old or the new number?
inc = revised[revised.cqi.notna()]
print('revised providers that CQI includes:', len(inc), '| CQI equals FINAL:', int((inc.cqi == inc.att_final).sum()), '| CQI equals PROVISIONAL:', int((inc.cqi == inc.att_prov).sum()))
print('all CQI provider-months in these six months: equals final', int(res.cqi_equals_final.sum()), 'of', int(res.cqi_providers.sum()), '| equals provisional', int(res.cqi_equals_prov.sum()))

revised providers that CQI includes: 9 | CQI equals FINAL: 9 | CQI equals PROVISIONAL: 0
all CQI provider-months in these six months: equals final 918 of 918 | equals provisional 909


In [6]:
res.to_parquet(c.GOLD / 'provisional_vs_final_monthly.parquet', index=False)
revised.to_parquet(c.GOLD / 'provisional_vs_final_revised_providers.parquet', index=False)